In [1]:
import sys
import os

python_path = sys.executable
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [3]:
spark = (
    SparkSession.builder.appName('Pyspark_intro2')
    .master('local[4]')
    .config('spark.pyspark.python', python_path)
    .config('spark.pyspark.driver.python', python_path)
    .config('spark.python.use.daemon', 'false')
    .config('spark.python.worker.faulthandler.enabled', 'true')
    .getOrCreate()
)

In [ ]:
df = spark.read.parquet('data/potato_prices_india.parquet')

### 1) basic inspection

In [44]:
df.limit(5).show()

+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+
|arrival_date|commodity|variety| grade|country|         state|  district|              market|min_price|max_price|modal_price|
+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato| potato|   faq|  india|andhra pradesh|   chittor|           palamaner|   1000.0|   1200.0|     1100.0|
|  2026-04-21|   potato|  other|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-21|   potato| potato|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|
|  2026-04-21|   potato| potato|   faq|  india|    chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|
|  2026-04-21|   potato|  local|medium|  india|   chattisgarh|      durg|                durg|   1000.0|   1200

In [34]:
print(f'rows = {df.count()}')
print(f'cols = {len(df.columns)}')

rows = 1863539
cols = 11


In [42]:
list(df.schema)

[StructField('arrival_date', DateType(), True),
 StructField('commodity', StringType(), True),
 StructField('variety', StringType(), True),
 StructField('grade', StringType(), True),
 StructField('country', StringType(), True),
 StructField('state', StringType(), True),
 StructField('district', StringType(), True),
 StructField('market', StringType(), True),
 StructField('min_price', DoubleType(), True),
 StructField('max_price', DoubleType(), True),
 StructField('modal_price', DoubleType(), True)]

In [45]:
df.printSchema()

root
 |-- arrival_date: date (nullable = true)
 |-- commodity: string (nullable = true)
 |-- variety: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- market: string (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- modal_price: double (nullable = true)



In [ ]:
df.select('max_price', 'min_price', 'modal_price').describe().show()

+-------+------------------+-----------------+------------------+
|summary|         max_price|        min_price|       modal_price|
+-------+------------------+-----------------+------------------+
|  count|           1863539|          1863539|           1863539|
|   mean|1604.9050498379697|1345.570895817045|1486.1168374903878|
| stddev|  8883.83814729352|8286.281226605759| 8297.261246119935|
|    min|             150.0|             80.0|             150.0|
|    max|             1.2E7|           1.12E7|            1.12E7|
+-------+------------------+-----------------+------------------+



In [49]:
df.select('modal_price').summary().show()

+-------+------------------+
|summary|       modal_price|
+-------+------------------+
|  count|           1863539|
|   mean|1486.1168374903878|
| stddev| 8297.261246119935|
|    min|             150.0|
|    25%|             800.0|
|    50%|            1200.0|
|    75%|            1800.0|
|    max|            1.12E7|
+-------+------------------+

